# 2. Azure ML Fine-Tuning and Model Registration

Submit reproducible SLM fine-tuning as an Azure ML command job, track the experiment with MLflow, persist outputs in the workspace datastore, and register an immutable candidate model.

## Experimental design

The notebook is the control plane; `lib/train.py` is the remote execution unit. Training consumes a versioned data asset, masks prompt tokens from the loss, adapts attention and MLP projections with rank-stabilized PEFT-compatible settings, evaluates against the validation split, and writes a merged model to a named job output.

**Mandatory production controls**
- Use a dedicated GPU compute cluster with managed identity and minimum nodes set to zero.
- Pin the curated environment and data/model asset versions.
- Keep the test split sealed until Notebook 3. Registration here creates a candidate, not an automatic production promotion.

In [1]:
from datetime import datetime, timezone
from pathlib import Path
import sys

from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "lib").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "lib").exists():
    raise RuntimeError("Start this notebook from the repository or notebooks directory")
sys.path.insert(0, str(PROJECT_ROOT))
load_dotenv(PROJECT_ROOT / ".env")

from lib.azureml_ops import (
    create_training_environment,
    register_job_model,
    submit_finetuning_job,
)

from lib.config import AzureMLConfig

## Run configuration

Replace immutable asset versions deliberately. The Key Vault fields are names, not secret values. Leave both as `None` only when the base model is public or already cached.

In [ ]:
DATA_ASSET_NAME = "raft-instance-security"
DATA_ASSET_VERSION = "20260804.065418"
BASE_MODEL = "meta-llama/Llama-3.2-1B-Instruct"
COMPUTE_NAME = "gpu-cluster1"
MANAGED_IDENTITY_CLIENT_ID = "39793817-16f4-45f9-8181-b8e198144c49"
TRAIN_ENVIRONMENT_NAME = "raft-slm-finetuning"
TRAIN_ENVIRONMENT_VERSION = "2"
EXPERIMENT_NAME = "raft-llama32-finetuning"
REGISTERED_MODEL_NAME = "raft-llama32-1b"
REGISTERED_MODEL_VERSION = datetime.now(timezone.utc).strftime("%Y%m%d.%H%M%S")

## Connect and resolve immutable inputs

`DefaultAzureCredential` uses your Azure CLI identity locally and managed identity in Azure ML. No subscription IDs or credentials are stored in notebook output.

In [3]:
config = AzureMLConfig.from_env()
ml_client = config.create_ml_client()
data_asset = ml_client.data.get(DATA_ASSET_NAME, version=DATA_ASSET_VERSION)
print("Data input:", data_asset.id)
print("Compute:", ml_client.compute.get(COMPUTE_NAME).name)

Data input: /subscriptions/ff9fa810-9dbb-4085-9c75-10b2f491bace/resourceGroups/sriks-mlhub-mcaps/providers/Microsoft.MachineLearningServices/workspaces/sriks-aml-ws/data/raft-instance-security/versions/20260804.065418
Compute: gpu-cluster1


## Register the pinned training environment

The environment definition lives in `environments/train-conda.yml`. Increment its version whenever a dependency changes; never mutate an environment used by a completed experiment.

In [4]:
training_environment = create_training_environment(
    ml_client, TRAIN_ENVIRONMENT_NAME, TRAIN_ENVIRONMENT_VERSION
)
print("Environment:", training_environment.id)

Environment: /subscriptions/ff9fa810-9dbb-4085-9c75-10b2f491bace/resourceGroups/sriks-mlhub-mcaps/providers/Microsoft.MachineLearningServices/workspaces/sriks-aml-ws/environments/raft-slm-finetuning/versions/2


>> The above command submits a job for creating the environment. Ensure environment is successfully created before proceeding.

## Submit and observe the remote job

Azure ML snapshots source code and mounts the output on workspace storage. Transformers metrics, parameters, system metrics, validation loss, and perplexity are sent to the workspace MLflow tracking server.

In [5]:
import os
print(os.environ.get("AZURE_KEY_VAULT_NAME"))
print(os.environ.get("AZURE_KEY_VAULT_HF_TOKEN_SECRET_NAME"))

sriksamlkeyvault6b382e43
hftoken


In [6]:
submitted_job = submit_finetuning_job(
    ml_client=ml_client,
    data_asset=f"azureml:{DATA_ASSET_NAME}:{DATA_ASSET_VERSION}",
    environment=training_environment.id,
    compute=COMPUTE_NAME,
    experiment_name=EXPERIMENT_NAME,
    base_model=BASE_MODEL,
    display_name=f"raft-sft-slm-{REGISTERED_MODEL_VERSION}",
    key_vault_name=os.environ.get("AZURE_KEY_VAULT_NAME"),
    hf_token_secret_name=os.environ.get("AZURE_KEY_VAULT_HF_TOKEN_SECRET_NAME"),
    managed_identity_client_id=MANAGED_IDENTITY_CLIENT_ID,
)

print("Job name:", submitted_job.name)

ml_client.jobs.stream(submitted_job.name)

Class AutoDeleteSettingSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class AutoDeleteConditionSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class BaseAutoDeleteSettingSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class IntellectualPropertySchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class ProtectionLevelSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class BaseIntellectualPropertySchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Uploading raft-finetuning-slm (0.08 MB

Job name: ivory_cart_pydrl6bzrd
RunId: ivory_cart_pydrl6bzrd
Web View: https://ml.azure.com/runs/ivory_cart_pydrl6bzrd?wsid=/subscriptions/ff9fa810-9dbb-4085-9c75-10b2f491bace/resourcegroups/sriks-mlhub-mcaps/workspaces/sriks-aml-ws

Streaming user_logs/std_log.txt

Successfully retrieved Hugging Face token from Key Vault

Generating train split: 0 examples [00:00, ? examples/s]
Generating train split: 650 examples [00:00, 36697.05 examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]
Generating validation split: 80 examples [00:00, 16588.93 examples/s]

Fetching 4 files: 100%|██████████| 4/4 [01:49<00:00, 27.46s/it]

Loading checkpoint shards: 100%|██████████| 4/4 [00:09<00:00,  2.41s/it]

Tokenizing RAFT records: 100%|██████████| 650/650 [00:02<00:00, 231.79 examples/s]

Tokenizing RAFT records: 100%|██████████| 80/80 [00:00<00:00, 243.28 examples/s]

  0%|          | 0/20 [00:00<?, ?it/s]
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen 

JobException: Exception : 
 {
    "error": {
        "code": "UserError",
        "severity": null,
        "message": "Execution failed. User process 'python' exited with status code 1. Please check log file 'user_logs/std_log.txt' for error details. Error:              ^^^^^^^^^^^^^^^^^^^^^^^^^^^\n  File \"/azureml-envs/azureml_6e5492a233ac43afa264d9302f9a9dc4/lib/python3.12/site-packages/transformers/utils/deprecation.py\", line 172, in wrapped_func\n    return func(*args, **kwargs)\n           ^^^^^^^^^^^^^^^^^^^^^\n  File \"/azureml-envs/azureml_6e5492a233ac43afa264d9302f9a9dc4/lib/python3.12/site-packages/transformers/models/llama/modeling_llama.py\", line 841, in forward\n    loss = self.loss_function(logits=logits, labels=labels, vocab_size=self.config.vocab_size, **kwargs)\n           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^\n  File \"/azureml-envs/azureml_6e5492a233ac43afa264d9302f9a9dc4/lib/python3.12/site-packages/transfo

## Inspect MLflow lineage

Use the experiment view for learning curves and system utilization. Programmatic lookup below makes the association between Azure ML job and MLflow run explicit.

In [ ]:
import mlflow

workspace = ml_client.workspaces.get(config.workspace_name)
mlflow.set_tracking_uri(workspace.mlflow_tracking_uri)
runs = mlflow.search_runs(
    experiment_names=[EXPERIMENT_NAME],
    max_results=10,
    order_by=["start_time DESC"],
)
visible_columns = [
    column
    for column in runs.columns
    if column in {"run_id", "status"}
    or column.startswith("metrics.")
    or column.startswith("params.")
]
runs[visible_columns].head()

## Register the candidate model

Registration points directly to the named job output in Azure Storage, retaining job lineage without a local download/re-upload. The candidate remains unpromoted until held-out evaluation and deployment smoke tests pass in Notebook 3.

In [ ]:
completed_job = ml_client.jobs.get(submitted_job.name)
if completed_job.status != "Completed":
    raise RuntimeError(f"Training did not complete successfully: {completed_job.status}")

registered_model = register_job_model(
    ml_client=ml_client,
    job_name=submitted_job.name,
    model_name=REGISTERED_MODEL_NAME,
    version=REGISTERED_MODEL_VERSION,
)
print("Registered candidate:", registered_model.id)

## Handoff and approval evidence

Retain the data fingerprint, source commit, environment version, job name, MLflow run, model version, training/evaluation curves, and responsible approver. Cost, license acceptance, PII review, red-team results, and rollback ownership belong in the model card before production promotion.